## Convolutional GAN with Wasserstein loss

This notebook provides an implementation of a convolutional GAN using the Wasserstein loss and the MNIST dataset.

In [1]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os

In [2]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import RMSprop
# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as F
# tqdm
from tqdm.notebook import tqdm, trange

In [3]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Device: cuda:0


### Load and Prepare MNIST Dataset
MNIST is a built in dataset with torchvision and we load it here and prepare dataloaders for PyTorch.

In [4]:
transform = transforms.Compose([transforms.Pad((2, 2)), transforms.ToTensor()])
mnist_trainset = MNIST(root='./data', train=True, download=True, transform=transform)

In [5]:
train_batch_size = 256
train_loader = data.DataLoader(mnist_trainset, batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True)

In [6]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Generator class
The generator class represents the Generator network. The network has a noise vector as input and outputs images of size (3, 32, 32). The network uses four transpose convolutions, three of which have a stride of 2 to expand the image size back up to 32 x 32. The first three layers use Batch Normalization and ReLU activation. The final layer does not have batch normalization and concludes with a Sigmoid function that matches the image scaling to the range [0, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_size=32, no_channels=1):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            nn.ConvTranspose2d( latent_dim, out_size * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(out_size * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(out_size * 4, out_size * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size * 2, out_size, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size, no_channels, 4, 2, 1, bias=False),
            nn.Sigmoid(),
        )
        
    def forward(self, noise):
        return self.gen(noise)

### Build the Discriminator class

The discriminator class represents the Discriminator network that will be trained to determine if an image is generated or from the real data distribution. This is a standard classification network using three dense layers with Leaky ReLU activation (note the final Sigmoid is in the loss function).


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_size=32, no_channels=1, neg_slope=0.2):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            nn.Conv2d(no_channels, in_size, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_size, in_size * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(in_size * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_size * 2, in_size * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(in_size * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_size * 4, 1, 4, 1, 0, bias=False),
        )
        
    def forward(self, image):
        return self.disc(image)

### Train the Model
The loss functions must be defined first and then the training loop can be run.

In [ ]:
def gen_loss(gen, disc, batch_size, latent_dim, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise)
    fake_pred = disc(fake_im)
    gen_loss = -fake_pred.mean()
    return gen_loss

In [ ]:
def disc_loss(gen, disc, real_im, batch_size, latent_dim, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise).detach()
    fake_pred = disc(fake_im)
    fake_loss = fake_pred.mean()
    real_pred = disc(real_im)
    real_loss = -real_pred.mean()
    disc_loss = fake_loss + real_loss
    return disc_loss

In [ ]:
n_epochs = 50
latent_dim = 100
lr = 0.00005
epoch_show = 10
show_batch = 64
n_critic = 5
weight_cliping_limit = 0.01

In [ ]:
gen = Generator(latent_dim=latent_dim).to(device)
gen_opt = RMSprop(gen.parameters(), lr=lr)
disc = Discriminator().to(device) 
disc_opt = RMSprop(disc.parameters(), lr=lr)

In [ ]:
gen_loss_lst = list()
disc_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_gen_loss = 0.0
    avg_disc_loss = 0.0
    num_items = 0
    for real_im, _ in tqdm(train_loader):
        
        for dsteps in range(n_critic):
            cur_batch_size = len(real_im)
            real_im = real_im.to(device)

            # Update discriminator
            disc_opt.zero_grad()
        
            # Clip parameter weights
            for p in disc.parameters():
                p.data.clamp_(-weight_cliping_limit, weight_cliping_limit)
        
            dloss = disc_loss(gen, disc, real_im, cur_batch_size, 
                              latent_dim, device)
            dloss.backward(retain_graph=True)
            disc_opt.step()

        # Update Generator
        gen_opt.zero_grad()
        gloss = gen_loss(gen, disc, cur_batch_size, 
                         latent_dim, device)
        gloss.backward(retain_graph=True)
        gen_opt.step()

        avg_gen_loss += gloss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if epoch % epoch_show == 0:
        noise = torch.randn(show_batch, latent_dim, 1, 1, device=device)
        fake_im = gen(noise)
        show_tensor_images(fake_im, filename='WGANMNISTClipGrid_' + str(epoch) + '.png')
        show_tensor_images(real_im)
        
    ave_gen_loss = avg_gen_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    gen_loss_lst.append(ave_gen_loss)
    disc_loss_lst.append(ave_disc_loss)
    tqdm_epoch.set_description('Ave Gen Loss: {:5f}, \
                               Ave Disc Loss: {:5f}'.format(ave_gen_loss, ave_disc_loss))
    torch.save(gen.state_dict(), 'wganmnistclip_genckpt.pth')
    torch.save(disc.state_dict(), 'wganmnistclip_discckpt.pth')

### Plot Losses

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, gen_loss_lst, label='Generator Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('WGANMNISTClipLosses.png', dpi=300, bbox_inches='tight')

In [ ]:
noise = torch.randn(40, latent_dim, 1, 1, device=device)
samples = gen(noise)
plt.figure(figsize=(10,20))
show_tensor_images(samples, nrow=20, filename='WGANMNISTClipGrid.png')

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(real_im, filename='ReWGANMNISTClipGrid.png')